# XGBoost Regressor training/validation

In [35]:
from xgboost import XGBRegressor, XGBRFRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import polars as pl
from skopt import BayesSearchCV
import pickle

import numpy as np

import random
from helpers.plotting import *

DATA_PATH = "./data/"
MODELS_PATH = "./models/"
SUBMISSIONS_PATH = "./submissions/"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [26]:
training_data = pl.read_csv(DATA_PATH + "label_encoded_for_trees.csv")
test_data = pl.read_csv(DATA_PATH + "test.csv")

## Baseline XGB

In [ ]:
X = training_data.drop("Listening_Time_minutes")
y = training_data["Listening_Time_minutes"]

baseline_xgb = XGBRegressor(random_state=RANDOM_SEED)

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error"
}

cv_baseline_xgb_res = cross_validate(baseline_xgb, X, y, cv=kfold, scoring=scoring, n_jobs=-1, return_estimator=True)

rmse_scores_baseline = -cv_baseline_xgb_res["test_rmse"]
r2_scores_baseline = cv_baseline_xgb_res["test_r2"]

print("Average R²:", r2_scores_baseline.mean())
print("Average RMSE:", rmse_scores_baseline.mean())

Average R²: 0.7684765577316284
Average RMSE: 13.057924461364745


## Baseline XGBRF

In [ ]:
baseline_xgbrf = XGBRFRegressor(random_state=RANDOM_SEED)

cv_baseline_xgbrf_res = cross_validate(baseline_xgbrf, X, y, cv=kfold, scoring=scoring, n_jobs=-1, return_estimator=True)

rmse_scores_baseline = -cv_baseline_xgbrf_res["test_rmse"]
r2_scores_baseline = cv_baseline_xgbrf_res["test_r2"]

print("Average R²:", r2_scores_baseline.mean())
print("Average RMSE:", rmse_scores_baseline.mean())

Average R²: 0.7613932132720947
Average RMSE: 13.256178665161134


## Hyperparameter tuning XGBRegressor

In [9]:
search_space = {
    "model__n_estimators": (50, 500),
    "model__learning_rate": (0.01, 0.3, "log-uniform"),
    "model__max_depth": (3, 10),
    "model__min_child_weight": (1, 10),
    "model__subsample": (0.5, 1.0, "uniform"),
    "model__colsample_bytree": (0.5, 1.0, "uniform"),
    "model__gamma": (0, 5.0),
    "model__reg_alpha": (0.0, 1.0),
    "model__reg_lambda": (0.0, 1.0)
}

In [ ]:
tuning_xgb = XGBRegressor(objective="reg:squarederror", random_state=RANDOM_SEED, n_jobs=-1)

# Set up Bayesian optimization
opt = BayesSearchCV(
    estimator=tuning_xgb,
    search_spaces=search_space,
    scoring="neg_root_mean_squared_error",
    n_iter=50,
    cv=kfold,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=0
)

opt.fit(X, y)

print("Best RMSE:", -opt.best_score_)
print("Best Params:", opt.best_params_)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

In [17]:
best_search_xgb = opt.best_estimator_

best_search_xgb_file = MODELS_PATH + "best_search_xgb.pkl"
with open(best_search_xgb_file, 'wb') as file:
    pickle.dump(best_search_xgb, file)

In [19]:
best_search_xgb.fit(X, y)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsample_bynode=None,
                              colsample_bytree=1.0, device=None,
                              early_stopping_rounds=None,
                              enable_categorical=False, eval_metric=None,
                              feature_types=None, feature_weights=None,
                              gamma=0.0, grow_policy=None, importance_type=None,
                              interaction_constraints=None,
                              learning_rate=0.05105022083626219, max_bin=None,
                              max_cat_threshold=None, max_cat_to_onehot=None,
                              max_delta_step=None, max_depth=10,
                              max_leaves=None, min_child_weight=1, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=-1,
                              num_parallel_tree=None, ...))])

In [29]:
def preprocessing_for_trees(df: pl.DataFrame) -> pl.DataFrame:
    dropped_id_and_title = df.drop("id", "Episode_Title")
    encoded_cats = dropped_id_and_title.with_columns(
        pl.col("Genre").cast(pl.Categorical).to_physical(),
        pl.col("Podcast_Name").cast(pl.Categorical).to_physical(),
        pl.col("Publication_Day").cast(pl.Categorical).to_physical(),
        pl.col("Publication_Time").cast(pl.Categorical).to_physical(),
        pl.col("Episode_Sentiment").cast(pl.Categorical).to_physical(),
    )

    return encoded_cats

In [30]:
prepared_test = preprocessing_for_trees(test_data)

In [31]:
y_pred_xgb_search = best_search_xgb.predict(prepared_test)

In [34]:
y_pred_xgb_search

xgb_search_submission = pl.DataFrame({
    "id": test_data["id"],
    "Listening_Time_minutes": y_pred_xgb_search
})

xgb_search_submission

id,Listening_Time_minutes
i64,f32
750000,59.063515
750001,17.313217
750002,51.309368
750003,72.962242
750004,48.915249
…,…
999995,12.052287
999996,58.846142
999997,6.996723


In [36]:
xgb_search_submission.write_csv(SUBMISSIONS_PATH + "best_xgb_search_submission.csv")